In [ ]:
from datetime import datetime, timedelta

"""
aantal decimalen
schrikkeljaar
wisselende rentepercentages
renteberekening per datum (ook toekomst)
eerste dag per opening rekening / renteberekening per
"""

class Account:
    def __init__(self, account_number, owner, initial_balance=0, interest_rate=0, opening_date=None):
        self.account_number = account_number
        self.owner = owner
        self.balance = initial_balance
        self.interest_rate = interest_rate / 100  # Convert annual rate to a decimal
        opening_date = datetime.strptime(opening_date, "%Y-%m-%d") if opening_date else datetime.now()
        self.transactions = [("Initial balance", initial_balance, opening_date, 0)]  # Initial transaction with a sequence number
        self.locked = False
        self.transaction_counter = 1  # Starts from 1 as 0 is used for the initial balance

    def print_transactions(self):
        print(f"{'Type':<15} {'Amount':<10} {'Date':<15} {'Days Since Last':<15} {'Balance':<10} {'Cumulative Interest':<20}")
        print("-" * 85)  # Adjusted width for the new columns

        # Initialize variables
        rolling_balance = self.transactions[0][1]
        last_date = self.transactions[0][2]
        total_interest = 0.0

        for i, transaction in enumerate(self.transactions):
            transaction_type, amount, date, sequence = transaction

            # Calculate days since last transaction for all but the first transaction
            days_since_last = (date - last_date).days if i > 0 else 0
            # Calculate interest since last transaction
            daily_rate = self.interest_rate / 365
            interest = rolling_balance * daily_rate * days_since_last
            total_interest += interest

            if transaction_type in ["Withdrawal", "Transfer out"]:
                rolling_balance -= amount
            elif transaction_type in ["Deposit", "Transfer in"]:
                rolling_balance += amount

            print(f"{transaction_type:<15} {amount:<10.2f} {date.strftime('%Y-%m-%d'):<15} {days_since_last:<15} {rolling_balance:<10.2f} {total_interest:<20.2f}")

            last_date = date  # Update last_date for the next iteration


    def add_transaction(self, transaction_type, amount, transaction_date):
        """Helper function to add transactions with a unique sequence number."""
        self.transactions.append((transaction_type, amount, transaction_date, self.transaction_counter))
        self.transaction_counter += 1

    def withdraw(self, amount, transaction_date=None):
        if self.locked:
            return "Account is locked."
        if amount <= 0 or amount > self.balance:
            return "Invalid amount."
        date = datetime.strptime(transaction_date, "%Y-%m-%d") if transaction_date else datetime.now()
        # Ensure each withdrawal is recorded uniquely
        self.add_transaction("Withdrawal", amount, date)
        self.balance -= amount
        return f"Withdrew {amount}. New balance: {self.balance}."
    
    def calculate_interest(self):
        if self.locked:
            return "Account is locked."

        # Sort transactions first by date and then by sequence number to ensure correct order
        sorted_transactions = sorted(self.transactions, key=lambda x: (x[2], x[3]))

        interest_details = []  # Holds the interest calculation steps
        total_interest = 0.0
        last_balance = sorted_transactions[0][1]
        last_date = sorted_transactions[0][2]

        for _, amount, transaction_date, _ in sorted_transactions[1:]:
            days = (transaction_date - last_date).days
            daily_rate = self.interest_rate / 365
            period_interest = last_balance * daily_rate * days
            total_interest += period_interest
            interest_details.append({
                "rekening": self.account_number,
                "rentepercentage": self.interest_rate * 100,
                "datum": transaction_date.strftime("%Y-%m-%d"),
                "saldo": last_balance,
                "rente": period_interest
            })
            # Update balance for next period's calculation
            # Assuming correct balance tracking logic here
            last_balance = self.balance
            last_date = transaction_date

        # Include interest calculation from the last transaction to the current date
        days = (datetime.now() - last_date).days
        period_interest = last_balance * daily_rate * days
        total_interest += period_interest
        interest_details.append({
            "rekening": self.account_number,
            "rentepercentage": self.interest_rate * 100,
            "datum": datetime.now().strftime("%Y-%m-%d"),
            "saldo": last_balance,
            "rente": period_interest
        })

        return total_interest, interest_details
        
    def deposit(self, amount, transaction_date=None):
        if self.locked:
            return "Account is locked."
        if amount <= 0:
            return "Invalid amount."
        date = datetime.strptime(transaction_date, "%Y-%m-%d") if transaction_date else datetime.now()
        self.balance += amount
        self.add_transaction("Deposit", amount, date)
        return f"Deposited {amount}. New balance: {self.balance}."

    def transfer(self, target_account, amount, transaction_date=None):
        if self.locked:
            return "Account is locked."
        if amount <= 0 or amount > self.balance:
            return "Invalid amount."
        date = datetime.strptime(transaction_date, "%Y-%m-%d") if transaction_date else datetime.now()
        # Each transfer, even if identical in amount and date, should be distinct
        self.add_transaction("Transfer out", amount, date)
        self.balance -= amount
        target_account.add_transaction("Transfer in", amount, date)
        target_account.balance += amount
        return f"Transferred {amount} to {target_account.account_number}. New balance: {self.balance}."


    def get_balance(self):
        return f"Balance: {self.balance}."

    def search_transactions(self, search_term):
        return [transaction for transaction in self.transactions if search_term in transaction[0]]

    def update_owner(self, new_owner):
        self.owner = new_owner
        return "Owner updated."

    def lock_account(self):
        self.locked = True
        return "Account locked."

    def unlock_account(self):
        self.locked = False
        return "Account unlocked."


In [ ]:
rek123 = Account(123, 'Rob', 1000, 2, '2024-01-01')
rek456 = Account(456, 'Lea', 2000, 1, '2024-01-01')

In [ ]:
print(rek123.withdraw(500, '2025-01-01'))
print(rek123.withdraw(500, '2026-01-01'))
print(rek456.withdraw(130, '2024-12-31'))

In [ ]:
rek123.print_transactions()
rek456.print_transactions()